# Lesson 3: Prompt Development

See [docs/03-prompt-development.md](../docs/03-prompt-development.md).

In [ ]:
import sys
from functools import partial
sys.path.insert(0, '..')

from src.init_env import set_environment_variables
from src.data_loader import load_mails, split_dataset, build_option_lists
from src.send_request import init_orchestration_service, send_request
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template

set_environment_variables()
init_orchestration_service()

EXAMPLE_MESSAGE_IDX = 10
mails = load_mails()
dev_set, test_set, test_set_small = split_dataset(mails)
categories, urgency, sentiment, option_lists = build_option_lists(mails)
mail = dev_set[EXAMPLE_MESSAGE_IDX]

In [ ]:
prompt_1 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Your task is to extract:
- urgency
- sentiment
Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_1 = partial(send_request, prompt=prompt_1)
response = f_1(input=mail['message'])

In [ ]:
prompt_2 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Your task is to extract:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_2 = partial(send_request, prompt=prompt_2, **option_lists)
response = f_2(input=mail['message'])

In [ ]:
prompt_3 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above.
Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_3 = partial(send_request, prompt=prompt_3, **option_lists)
response = f_3(input=mail['message'])

In [ ]:
prompt_4 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.

Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_4 = partial(send_request, prompt=prompt_4, **option_lists)
response = f_4(input=mail['message'])

In [ ]:
prompt_8 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.

Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_8 = partial(send_request, prompt=prompt_8, **option_lists)
response = f_8(input=mail['message'])